In [ ]:
import hashlib
import os
import time
import matplotlib.pyplot as plt
import numpy as np

# --- 1. CONFIGURATION ---
HASH_BITS = 40
HASH_BYTES = HASH_BITS // 8  # 5 bytes
NUM_EXPERIMENTS = 100       # We will run 100 experiments for each to get a good average

def get_truncated_hash(message_bytes):
    """SHA-256 truncated to 40-bits (5 bytes)"""
    return hashlib.sha256(message_bytes).digest()[:HASH_BYTES]

def next_state(current_bytes):
    """Deterministic step function: takes a 5-byte hash and yields the next 5-byte hash."""
    return get_truncated_hash(current_bytes)


# --- 2. NAIVE COLLISION SEARCH (High Memory, O(1) Lookup) ---
def run_naive_search():
    seen = {}
    attempts = 0
    # Start with a random 5-byte seed
    current = os.urandom(HASH_BYTES)
    
    while True:
        attempts += 1
        nxt = next_state(current)
        
        if nxt in seen:
            # Collision found!
            return attempts, seen[nxt], current, nxt
            
        seen[nxt] = current
        current = nxt


# --- 3. LOW-MEMORY COLLISION SEARCH (Floyd's Tortoise and Hare) ---
def run_low_memory_search():
    # Start both pointers at the same random 5-byte seed
    start_seed = os.urandom(HASH_BYTES)
    tortoise = start_seed
    hare = start_seed
    
    attempts = 0
    
    # Phase 1: Finding a cycle (where tortoise and hare meet)
    while True:
        tortoise = next_state(tortoise)       # 1 step
        hare = next_state(next_state(hare))   # 2 steps
        attempts += 1
        
        if tortoise == hare:
            break

    # Phase 2: Finding the exact start of the collision
    tortoise = start_seed
    while True:
        next_tortoise = next_state(tortoise)
        next_hare = next_state(hare)
        
        if next_tortoise == next_hare:
            # We found the exact messages that collide!
            return attempts, tortoise, hare, next_tortoise
            
        tortoise = next_tortoise
        hare = next_hare


# --- 4. THE EXPERIMENT RUNNER ---
def run_benchmark():
    naive_attempts = []
    low_mem_attempts = []

    print(f"🚀 Running {NUM_EXPERIMENTS} experiments for Naive and Low-Memory...")
    
    for i in range(NUM_EXPERIMENTS):
        # Run Naive
        n_att, _, _, _ = run_naive_search()
        naive_attempts.append(n_att)

        # Run Low Memory
        lm_att, _, _, _ = run_low_memory_search()
        low_mem_attempts.append(lm_att)

        if (i + 1) % 10 == 0:
            print(f"Completed {i + 1}/{NUM_EXPERIMENTS} runs...")

    return naive_attempts, low_mem_attempts

# Run the benchmarks!
naive_data, low_mem_data = run_benchmark()

# --- 5. DATA ANALYSIS & PLOTTING ---
avg_naive = np.mean(naive_data)
avg_low_mem = np.mean(low_mem_data)

# Theoretical Birthday Bound is ~ 2^(40/2) = 2^20 = 1,048,576
theoretical_bound = 2**(HASH_BITS / 2)

print("\n--- STATISTICAL RESULTS ---")
print(f"Theoretical Birthday Bound: {theoretical_bound:,.0f} steps")
print(f"Average Naive Attempts: {avg_naive:,.2f} steps")
print(f"Average Low-Mem Attempts (Cycle detection): {avg_low_mem:,.2f} steps")

# Plotting the histograms
plt.figure(figsize=(10, 6))
plt.hist(naive_data, bins=20, alpha=0.6, label='Naive Search Attempts', color='blue')
plt.hist(low_mem_data, bins=20, alpha=0.6, label='Low-Memory Attempts', color='orange')
plt.axvline(avg_naive, color='darkblue', linestyle='dashed', linewidth=2, label=f'Avg Naive ({avg_naive:.0f})')
plt.axvline(avg_low_mem, color='darkorange', linestyle='dashed', linewidth=2, label=f'Avg Low-Mem ({avg_low_mem:.0f})')
plt.axvline(theoretical_bound, color='red', linestyle='dotted', linewidth=2, label='Theoretical Bound ($2^{20}$)')

plt.title('Distribution of Collision Search Attempts (Naive vs Low-Memory)')
plt.xlabel('Number of Attempts to find Collision')
plt.ylabel('Frequency (Out of 100 experiments)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

🚀 Running 100 experiments for Naive and Low-Memory...
Completed 10/100 runs...
